# BM25 Experiment Summary

## Initial Result:

The first BM25 evaluation gave unexpectedly low scores: **MAP = 0.00871, MRR = 0.02068, P@5 = 0.00978, P@10 = 0.00667, nDCG@10 = 0.00607.**

## Investigation:

We checked the BM25 results and qrels for possible formatting, data-type, scoring, and matching issues. The `qid` and `docno` data types were compatible. We also manually checked Query 1 and found that BM25 was retrieving relevant documents near the top, indicating that the BM25 scoring and retrieval process were working correctly.

The main issue was the query ID assignment. The numbers after the `.I` tag in `cran.qry` are not the IDs used for matching with the qrels in our setup. For example, `cran.qry` contains `.I 001`, `.I 002`, `.I 004`, `.I 008`, etc., while the qrels use sequential query IDs based on the position of the query blocks.

Therefore, query IDs were reassigned according to their position in `cran.qry`:

1st query → qid 1

2nd query → qid 2

3rd query → qid 3

...

225th query → qid 225

The actual number after `.I` was ignored.

## Important clarification:

BM25 scores and qrel relevance labels are different things. BM25 produces scores to rank documents, while qrels provide the ground-truth relevance labels. The scores are not expected to match the relevance grades.

After correcting the query ID alignment, the BM25 evaluation improved substantially:

**MAP = 0.315701**

**MRR = 0.544207**

**P@5 = 0.329778**

**P@10 = 0.237333**

**nDCG@10 = 0.343508**

**Index Time ≈ 3.84 seconds**

**Search Time ≈ 5.32 seconds**

## Final Finding:

The extremely low initial BM25 scores were mainly caused by incorrect query-to-qrels ID alignment rather than a problem with BM25 scoring or data types. After fixing the query IDs, the retrieval results aligned correctly with the qrels. The corrected BM25 result can now be used for comparison with the other retrieval models.

The BM25 implementation in the notebook uses PyTerrier's `Retriever` with `wmodel="BM25"`, while keeping the same indexed collection and preprocessing pipeline.


Environment Setup

In [ ]:
%pip install -q python-terrier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.1/223.1 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 27.9 MB/s eta 0:00:00


In [ ]:
import pyterrier as pt
import pandas as pd
import re
import time

if not pt.started():
    pt.init()

print("PyTerrier version:", pt.__version__)

/tmp/ipykernel_858/1066108698.py:6: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done
PyTerrier version: 1.1.2


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_858/1066108698.py:7: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


Obtain Cranfield collection

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cran.tar.gz to cran.tar.gz


In [ ]:
import tarfile
with tarfile.open("cran.tar.gz", "r:gz") as cran:
  cran.extractall("cranfield")

/tmp/ipykernel_858/1663252741.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  cran.extractall("cranfield")


Preprocessing of Documents

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving stopwords.txt to stopwords.txt
Saving outliers_preprocess.py to outliers_preprocess.py
Saving outliers_porter.py to outliers_porter.py


In [ ]:
import sys

sys.path.append("/content")

In [ ]:
from outliers_preprocess import (
    parse_documents,
    preprocess_text,
    load_stemmed_stopwords
)

from outliers_porter import PorterStemmer

In [ ]:
from pathlib import Path

input_path = Path("/content/cranfield/cran.all.1400")
stopwords_path = Path("/content/stopwords.txt")

In [ ]:
stemmer = PorterStemmer()
stemmed_stopwords = load_stemmed_stopwords(stopwords_path, stemmer)
documents = parse_documents(input_path)

In [ ]:
processed_documents = []

for doc_id, text in documents:
    tokens = preprocess_text(text, stemmer, stemmed_stopwords)

    processed_documents.append({
        "docno": str(doc_id),
        "text": " ".join(tokens)
    })

In [ ]:
docs = pd.DataFrame(processed_documents)
print(docs.head())

  docno                                               text
0     1  experiment investig aerodynam wing slipstream ...
1     2  simpl shear flow past flat plate incompress fl...
2     3  boundari layer simpl shear flow past flat plat...
3     4  approxim solut incompress laminar boundari lay...
4     5  dimension transient heat conduct doubl layer s...


Preprocessing of Queries

In [ ]:
def parse_queries(file_path):

    queries = []

    with open(file_path, "r", encoding="ascii", errors="ignore") as f:
        lines = f.readlines()

    current_qid = None
    current_query = []
    in_query = False

    for line in lines:

        line = line.rstrip("\n")

        # Query ID
        match = re.match(r"\.I\s+(\d+)", line)

        if match:

            # Save previous query
            if current_qid is not None:

                queries.append({
                    "qid": str(int(current_qid)),
                    "query": " ".join(current_query).strip()
                })

            current_qid = match.group(1)
            current_query = []
            in_query = False

        elif line.strip() == ".W":

            in_query = True

        elif in_query:

            current_query.append(line.strip())

    # Save last query
    if current_qid is not None:

        queries.append({
            "qid": current_qid,
            "query": " ".join(current_query).strip()
        })

    return pd.DataFrame(queries)


In [ ]:
def parse_queries(file_path):

    queries = []

    with open(file_path, "r", encoding="ascii", errors="ignore") as f:
        lines = f.readlines()

    current_query = []
    in_query = False

    for line in lines:

        line = line.rstrip("\n")

        # Start of a new query block
        if re.match(r"\.I\s+\d+", line):

            # Save previous query
            if current_query:
                queries.append({
                    "qid": str(len(queries) + 1),
                    "query": " ".join(current_query).strip()
                })

            current_query = []
            in_query = False

        elif line.strip() == ".W":

            in_query = True

        elif in_query:

            current_query.append(line.strip())

    # Save last query
    if current_query:
        queries.append({
            "qid": str(len(queries) + 1),
            "query": " ".join(current_query).strip()
        })

    return pd.DataFrame(queries)

In [ ]:
query = parse_queries("/content/cranfield/cran.qry")

In [ ]:
processed_queries = []

for _, row in query.iterrows():

    tokens = preprocess_text(row["query"],stemmer,stemmed_stopwords)

    processed_queries.append({
        "qid": str(row["qid"]),
        "query": " ".join(tokens)
    })

query = pd.DataFrame(processed_queries)

In [ ]:
display(query.head(10))

,qid,query
0,1,similar law obey construct aeroelast model hea...
1,2,structur aeroelast problem associ flight high ...
2,3,problem heat conduct composit slab solv far
3,4,criterion develop empir valid flow solut chemi...
4,5,chemic kinet applic hyperson aerodynam problem
5,6,theoret experiment guid turbul couett flow beh...
6,7,possibl relat avail pressur distribut ogiv for...
7,8,method dash exact approxim dash present avail ...
8,9,paper intern slip flow heat transfer studi
9,10,real ga transport properti air avail wide rang...


Indexing

In [ ]:
index_path = "/content/cranfield_tfidf_index"

indexer = pt.index.IterDictIndexer(
    index_path,
    meta=["docno"],
    text_attrs=["text"],
    overwrite=True
)

In [ ]:
start_time = time.perf_counter()

indexref = indexer.index(
    docs.to_dict("records")
)

index_time = time.perf_counter() - start_time

print(f"Indexing time: {index_time:.4f} seconds")

17:43:18.223 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (471) - further warnings are suppressed
17:43:20.002 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
Indexing time: 3.8436 seconds


In [ ]:
index = pt.IndexFactory.of(indexref)

In [ ]:
print(index.getCollectionStatistics())

Number of documents: 1400
Number of terms: 4389
Number of postings: 77353
Number of fields: 0
Number of tokens: 132181
Field names: []
Positions:   false



In [ ]:
print(index.getLexicon())

<org.terrier.structures.Lexicon at 0x79b7cc13e670 jclass=org/terrier/structures/Lexicon jself=<LocalRef obj=0x3e0b5c82 at 0x79b7cc31af30>>


BM25

In [ ]:
bm25 = pt.terrier.Retriever(
    index,
    wmodel="BM25"
)

In [ ]:
start = time.perf_counter()

bm25_results = bm25.transform(query)

search_time = time.perf_counter() - start

print(f"Search time: {search_time:.4f} seconds")

Search time: 5.3160 seconds


In [ ]:
bm25_results

,qid,query,docid,docno,rank,score
0,1,similar law obey construct aeroelast model hea...,50,51,0,29.524889
1,1,similar law obey construct aeroelast model hea...,485,486,1,28.757999
2,1,similar law obey construct aeroelast model hea...,183,184,2,25.137115
3,1,similar law obey construct aeroelast model hea...,11,12,3,24.962690
4,1,similar law obey construct aeroelast model hea...,877,878,4,22.776485
...,...,...,...,...,...,...
188970,225,design factor control lift drag ratio mach num...,836,837,891,0.431656
188971,225,design factor control lift drag ratio mach num...,1143,1144,892,0.421392
188972,225,design factor control lift drag ratio mach num...,82,83,893,0.419965
188973,225,design factor control lift drag ratio mach num...,1391,1392,894,0.414354


Evaluation

In [ ]:
qrels = pd.read_csv(
    "/content/cranfield/cranqrel",
    sep=r"\s+",
    header=None,
    names=["qid", "docno", "label"]
)

qrels["qid"] = qrels["qid"].astype(str)
qrels["docno"] = qrels["docno"].astype(str)

print(qrels.head())
print("Number of relevance judgments:", len(qrels))
print("Number of queries:", qrels["qid"].nunique())

  qid docno  label
0   1   184      2
1   1    29      2
2   1    31      2
3   1    12      3
4   1    51      3
Number of relevance judgments: 1837
Number of queries: 225


In [ ]:
print("BM25 columns:")
print(bm25_results.columns.tolist())

print("\nQrels columns:")
print(qrels.columns.tolist())

BM25 columns:
['qid', 'query', 'docid', 'docno', 'rank', 'score']

Qrels columns:
['qid', 'docno', 'label']


In [ ]:
print(bm25_results[["qid", "docno"]].dtypes)
print(qrels[["qid", "docno"]].dtypes)

qid      object
docno    object
dtype: object
qid      object
docno    object
dtype: object


In [ ]:
bm25_eval = pt.Evaluate(
    bm25_results,
    qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)

bm25_eval

{'map': 0.31570062120604253,
 'recip_rank': 0.5442072803431575,
 'P.5': 0.329777777777778,
 'P.10': 0.2373333333333335,
 'ndcg_cut.10': 0.3435079529364762}

In [ ]:
bm25_results_table = pd.DataFrame([{
    "Model": "BM25",
    "MAP": bm25_eval["map"],
    "MRR": bm25_eval["recip_rank"],
    "P@5": bm25_eval["P.5"],
    "P@10": bm25_eval["P.10"],
    "nDCG@10": bm25_eval["ndcg_cut.10"],
    "Index Time (s)": index_time,
    "Search Time (s)": search_time
}])

bm25_results_table

,Model,MAP,MRR,P@5,P@10,nDCG@10,Index Time (s),Search Time (s)
0,BM25,0.315701,0.544207,0.329778,0.237333,0.343508,3.843604,5.315987


Parameter Fine Tuning

In [ ]:
# import pandas as pd
# import time

# bm25_tuning_results = []

# k1_values = [0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]
# b_values = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]

# for k1 in k1_values:
#     for b in b_values:

#         bm25_model = pt.terrier.Retriever(
#             index,
#             wmodel="BM25",
#             controls={
#                 "bm25.k_1": str(k1),
#                 "bm25.k_3": "8",
#                 "bm25.b": str(b)
#             }
#         )

#         start = time.perf_counter()

#         results = bm25_model.transform(query)

#         search_time = time.perf_counter() - start

#         evaluation = pt.Evaluate(
#             results,
#             qrels,
#             metrics=[
#                 "map",
#                 "recip_rank",
#                 "P.5",
#                 "P.10",
#                 "ndcg_cut.10"
#             ]
#         )

#         bm25_tuning_results.append({
#             "k1": k1,
#             "b": b,
#             "MAP": evaluation["map"],
#             "MRR": evaluation["recip_rank"],
#             "P@5": evaluation["P.5"],
#             "P@10": evaluation["P.10"],
#             "nDCG@10": evaluation["ndcg_cut.10"],
#             "Search Time (s)": search_time
#         })

# bm25_tuning_df = pd.DataFrame(bm25_tuning_results)

# bm25_tuning_df.sort_values(
#     by="MAP",
#     ascending=False
# ).head(10)


import pandas as pd
import time

bm25_tuning_results = []

# =========================
# STAGE 1: COARSE SEARCH
# =========================

k1_values = [0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4]
b_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

print("Starting Stage 1: Coarse Search...")
print("Total combinations:", len(k1_values) * len(b_values))

for k1 in k1_values:
    for b in b_values:

        bm25_model = pt.terrier.Retriever(
            index,
            wmodel="BM25",
            controls={
                "bm25.k_1": str(k1),
                "bm25.k_3": "8",
                "bm25.b": str(b)
            }
        )

        start = time.perf_counter()

        results = bm25_model.transform(query)

        search_time = time.perf_counter() - start

        evaluation = pt.Evaluate(
            results,
            qrels,
            metrics=[
                "map",
                "recip_rank",
                "P.5",
                "P.10",
                "ndcg_cut.10"
            ]
        )

        bm25_tuning_results.append({
            "k1": k1,
            "b": b,
            "MAP": evaluation["map"],
            "MRR": evaluation["recip_rank"],
            "P@5": evaluation["P.5"],
            "P@10": evaluation["P.10"],
            "nDCG@10": evaluation["ndcg_cut.10"],
            "Search Time (s)": search_time
        })

# Create dataframe
bm25_tuning_df = pd.DataFrame(bm25_tuning_results)

# Best result from Stage 1
best_stage1 = bm25_tuning_df.sort_values(
    by=["MAP", "nDCG@10", "MRR"],
    ascending=False
).iloc[0]

best_k1 = best_stage1["k1"]
best_b = best_stage1["b"]

print("\nStage 1 completed.")
print("Best k1:", best_k1)
print("Best b :", best_b)
print("Best MAP:", best_stage1["MAP"])


# =========================
# STAGE 2: FINE SEARCH
# =========================

# Automatically create fine-grained values
k1_values_fine = [
    round(x, 2)
    for x in [
        best_k1 - 0.20,
        best_k1 - 0.15,
        best_k1 - 0.10,
        best_k1 - 0.05,
        best_k1,
        best_k1 + 0.05,
        best_k1 + 0.10,
        best_k1 + 0.15,
        best_k1 + 0.20
    ]
    if 0.1 <= x <= 3.0
]

b_values_fine = [
    round(x, 2)
    for x in [
        best_b - 0.15,
        best_b - 0.10,
        best_b - 0.05,
        best_b,
        best_b + 0.05,
        best_b + 0.10,
        best_b + 0.15
    ]
    if 0.0 <= x <= 1.0
]

print("\nStarting Stage 2: Fine Search...")
print("Fine k1 values:", k1_values_fine)
print("Fine b values :", b_values_fine)
print("Total combinations:", len(k1_values_fine) * len(b_values_fine))

for k1 in k1_values_fine:
    for b in b_values_fine:

        bm25_model = pt.terrier.Retriever(
            index,
            wmodel="BM25",
            controls={
                "bm25.k_1": str(k1),
                "bm25.k_3": "8",
                "bm25.b": str(b)
            }
        )

        start = time.perf_counter()

        results = bm25_model.transform(query)

        search_time = time.perf_counter() - start

        evaluation = pt.Evaluate(
            results,
            qrels,
            metrics=[
                "map",
                "recip_rank",
                "P.5",
                "P.10",
                "ndcg_cut.10"
            ]
        )

        bm25_tuning_results.append({
            "k1": k1,
            "b": b,
            "MAP": evaluation["map"],
            "MRR": evaluation["recip_rank"],
            "P@5": evaluation["P.5"],
            "P@10": evaluation["P.10"],
            "nDCG@10": evaluation["ndcg_cut.10"],
            "Search Time (s)": search_time
        })


# =========================
# FINAL RESULTS
# =========================

bm25_tuning_df = pd.DataFrame(bm25_tuning_results)

bm25_tuning_df = bm25_tuning_df.sort_values(
    by=["MAP", "nDCG@10", "MRR"],
    ascending=False
)

print("\nTop 10 BM25 configurations:")
display(bm25_tuning_df.head(10))

Starting Stage 1: Coarse Search...
Total combinations: 110


KeyboardInterrupt: 

In [ ]:
best_bm25 = bm25_tuning_df.sort_values(
    by="MAP",
    ascending=False
).iloc[0]

print("Best BM25 parameters:")
print("k1 =", best_bm25["k1"])
print("b  =", best_bm25["b"])

print("\nBest results:")
print("MAP =", best_bm25["MAP"])
print("MRR =", best_bm25["MRR"])
print("P@5 =", best_bm25["P@5"])
print("P@10 =", best_bm25["P@10"])
print("nDCG@10 =", best_bm25["nDCG@10"])
print("Search Time =", best_bm25["Search Time (s)"])

Best BM25 parameters:
k1 = 1.6
b  = 0.7

Best results:
MAP = 0.3190627975330171
MRR = 0.5447608079439897
P@5 = 0.329777777777778
P@10 = 0.2435555555555557
nDCG@10 = 0.3483846582333991
Search Time = 3.571965265000017


In [ ]:
best_k1 = best_bm25["k1"]
best_b = best_bm25["b"]

tuned_bm25 = pt.terrier.Retriever(
    index,
    wmodel="BM25",
    controls={
        "bm25.k_1": str(best_k1),
        "bm25.k_3": "8",
        "bm25.b": str(best_b)
    }
)

start = time.perf_counter()

tuned_bm25_results = tuned_bm25.transform(query)

tuned_search_time = time.perf_counter() - start

tuned_bm25_eval = pt.Evaluate(
    tuned_bm25_results,
    qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)

print("Tuned BM25")
print("k1 =", best_k1)
print("b  =", best_b)

print("\nMAP:", tuned_bm25_eval["map"])
print("MRR:", tuned_bm25_eval["recip_rank"])
print("P@5:", tuned_bm25_eval["P.5"])
print("P@10:", tuned_bm25_eval["P.10"])
print("nDCG@10:", tuned_bm25_eval["ndcg_cut.10"])
print("Search Time:", tuned_search_time)

Tuned BM25
k1 = 1.6
b  = 0.7

MAP: 0.3190627975330171
MRR: 0.5447608079439897
P@5: 0.329777777777778
P@10: 0.2435555555555557
nDCG@10: 0.3483846582333991
Search Time: 3.6396501880003598


In [ ]:
best_k1 = best_bm25["k1"]
best_b = best_bm25["b"]

tuned_bm25 = pt.terrier.Retriever(
    index,
    wmodel="BM25",
    controls={
        "bm25.k_1": str(best_k1),
        "bm25.k_3": "8",
        "bm25.b": str(best_b)
    }
)

start = time.perf_counter()

tuned_bm25_results = tuned_bm25.transform(query)

tuned_search_time = time.perf_counter() - start

tuned_bm25_eval = pt.Evaluate(
    tuned_bm25_results,
    qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)

print("Tuned BM25")
print("k1 =", best_k1)
print("b  =", best_b)

print("\nMAP:", tuned_bm25_eval["map"])
print("MRR:", tuned_bm25_eval["recip_rank"])
print("P@5:", tuned_bm25_eval["P.5"])
print("P@10:", tuned_bm25_eval["P.10"])
print("nDCG@10:", tuned_bm25_eval["ndcg_cut.10"])
print("Search Time:", tuned_search_time)

Tuned BM25
k1 = 1.6
b  = 0.7

MAP: 0.3190627975330171
MRR: 0.5447608079439897
P@5: 0.329777777777778
P@10: 0.2435555555555557
nDCG@10: 0.3483846582333991
Search Time: 4.316678351000064


In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "BM25 Default",
        "MAP": bm25_eval["map"],
        "MRR": bm25_eval["recip_rank"],
        "P@5": bm25_eval["P.5"],
        "P@10": bm25_eval["P.10"],
        "nDCG@10": bm25_eval["ndcg_cut.10"],
        "Index Time (s)": index_time,
        "Search Time (s)": search_time
    },
    {
        "Model": "BM25 Tuned",
        "MAP": tuned_bm25_eval["map"],
        "MRR": tuned_bm25_eval["recip_rank"],
        "P@5": tuned_bm25_eval["P.5"],
        "P@10": tuned_bm25_eval["P.10"],
        "nDCG@10": tuned_bm25_eval["ndcg_cut.10"],
        "Index Time (s)": index_time,
        "Search Time (s)": tuned_search_time
    }
])

comparison

,Model,MAP,MRR,P@5,P@10,nDCG@10,Index Time (s),Search Time (s)
0,BM25 Default,0.315701,0.544207,0.329778,0.237333,0.343508,3.843604,3.579989
1,BM25 Tuned,0.319063,0.544761,0.329778,0.243556,0.348385,3.843604,4.316678


In [ ]:
improvement = pd.DataFrame({
    "Metric": [
        "MAP",
        "MRR",
        "P@5",
        "P@10",
        "nDCG@10"
    ],
    "Default BM25": [
        bm25_eval["map"],
        bm25_eval["recip_rank"],
        bm25_eval["P.5"],
        bm25_eval["P.10"],
        bm25_eval["ndcg_cut.10"]
    ],
    "Tuned BM25": [
        tuned_bm25_eval["map"],
        tuned_bm25_eval["recip_rank"],
        tuned_bm25_eval["P.5"],
        tuned_bm25_eval["P.10"],
        tuned_bm25_eval["ndcg_cut.10"]
    ]
})

improvement["Improvement"] = (
    improvement["Tuned BM25"] -
    improvement["Default BM25"]
)

improvement

,Metric,Default BM25,Tuned BM25,Improvement
0,MAP,0.315701,0.319063,0.003362
1,MRR,0.544207,0.544761,0.000554
2,P@5,0.329778,0.329778,0.000000
3,P@10,0.237333,0.243556,0.006222
4,nDCG@10,0.343508,0.348385,0.004877


Fine Tuning new


In [ ]:
import pandas as pd
import numpy as np
import time

# ==========================================
# SMART RANDOMIZED BM25 SEARCH
# ==========================================

np.random.seed(42)

random_trials = 300

bm25_tuning_results = []

print("Starting Smart Randomized BM25 Search")
print("Number of random configurations:", random_trials)


# ------------------------------------------
# Generate diverse random configurations
# ------------------------------------------

random_configs = []

for i in range(random_trials):

    k1 = round(np.random.uniform(0.2, 3.0), 2)
    b = round(np.random.uniform(0.0, 1.0), 2)

    # Use a wider range but give more useful values
    # for short queries
    k3_choices = [
        0, 1, 2, 3, 4, 5, 6,
        8, 10, 12, 15, 20
    ]

    k3 = np.random.choice(k3_choices)

    random_configs.append((k1, b, k3))


# Remove duplicate configurations
random_configs = list(set(random_configs))

print("Unique configurations:", len(random_configs))


# ==========================================
# RUN RANDOM SEARCH
# ==========================================

for count, (k1, b, k3) in enumerate(random_configs, 1):

    bm25_model = pt.terrier.Retriever(
        index,
        wmodel="BM25",
        controls={
            "bm25.k_1": str(k1),
            "bm25.b": str(b),
            "bm25.k_3": str(k3)
        }
    )

    start = time.perf_counter()

    results = bm25_model.transform(query)

    search_time = time.perf_counter() - start

    evaluation = pt.Evaluate(
        results,
        qrels,
        metrics=[
            "map",
            "recip_rank",
            "P.5",
            "P.10",
            "ndcg_cut.10"
        ]
    )

    bm25_tuning_results.append({
        "k1": k1,
        "b": b,
        "k3": k3,
        "MAP": evaluation["map"],
        "MRR": evaluation["recip_rank"],
        "P@5": evaluation["P.5"],
        "P@10": evaluation["P.10"],
        "nDCG@10": evaluation["ndcg_cut.10"],
        "Search Time (s)": search_time
    })

    if count % 25 == 0:
        print(
            f"Completed {count}/{len(random_configs)} trials"
        )


# ==========================================
# RANDOM SEARCH RESULTS
# ==========================================

random_df = pd.DataFrame(bm25_tuning_results)

random_df = random_df.sort_values(
    by=["MAP", "nDCG@10", "MRR"],
    ascending=False
).reset_index(drop=True)


print("\n======================================")
print("TOP 20 RANDOM SEARCH CONFIGURATIONS")
print("======================================")

display(random_df.head(20))

Starting Smart Randomized BM25 Search
Number of random configurations: 300
Unique configurations: 300
Completed 25/300 trials
Completed 50/300 trials
Completed 75/300 trials
Completed 100/300 trials
Completed 125/300 trials
Completed 150/300 trials
Completed 175/300 trials
Completed 200/300 trials
Completed 225/300 trials
Completed 250/300 trials
Completed 275/300 trials
Completed 300/300 trials

TOP 20 RANDOM SEARCH CONFIGURATIONS


,k1,b,k3,MAP,MRR,P@5,P@10,nDCG@10,Search Time (s)
0,2.99,0.75,4,0.322693,0.543982,0.332444,0.244444,0.348424,4.571337
1,2.90,0.81,10,0.322164,0.541145,0.331556,0.245333,0.349326,4.230940
2,2.95,0.61,1,0.321504,0.541354,0.336000,0.247111,0.349971,3.858904
3,2.85,0.85,0,0.321441,0.540714,0.330667,0.244444,0.349056,4.446255
4,2.19,0.81,3,0.321390,0.544641,0.335111,0.243556,0.347133,3.568376
5,2.50,0.72,15,0.320933,0.544860,0.330667,0.245778,0.348495,3.976225
6,2.58,0.75,1,0.320791,0.543642,0.336000,0.246667,0.348694,4.744929
7,2.67,0.75,4,0.320612,0.542433,0.332444,0.246667,0.348138,3.939057
8,2.64,0.80,20,0.320602,0.541338,0.331556,0.244000,0.346960,3.584174
9,2.44,0.68,2,0.320538,0.544801,0.331556,0.244889,0.347204,3.625291


In [ ]:
# ==========================================
# LOCAL REFINEMENT AROUND TOP CONFIGURATIONS
# ==========================================

top_configs = random_df.head(10)

fine_configs = set()

for _, row in top_configs.iterrows():

    best_k1 = row["k1"]
    best_b = row["b"]
    best_k3 = int(row["k3"])

    # Fine variation around k1
    k1_range = [
        round(best_k1 - 0.10, 2),
        round(best_k1 - 0.05, 2),
        round(best_k1, 2),
        round(best_k1 + 0.05, 2),
        round(best_k1 + 0.10, 2)
    ]

    # Fine variation around b
    b_range = [
        round(best_b - 0.10, 2),
        round(best_b - 0.05, 2),
        round(best_b, 2),
        round(best_b + 0.05, 2),
        round(best_b + 0.10, 2)
    ]

    # Fine variation around k3
    k3_range = [
        max(0, best_k3 - 2),
        max(0, best_k3 - 1),
        best_k3,
        best_k3 + 1,
        best_k3 + 2
    ]

    for k1 in k1_range:
        for b in b_range:
            for k3 in k3_range:

                if 0.1 <= k1 <= 3.0 and 0.0 <= b <= 1.0:
                    fine_configs.add(
                        (k1, b, k3)
                    )


print("Fine configurations:", len(fine_configs))


# ==========================================
# RUN FINE SEARCH
# ==========================================

for count, (k1, b, k3) in enumerate(fine_configs, 1):

    bm25_model = pt.terrier.Retriever(
        index,
        wmodel="BM25",
        controls={
            "bm25.k_1": str(k1),
            "bm25.b": str(b),
            "bm25.k_3": str(k3)
        }
    )

    start = time.perf_counter()

    results = bm25_model.transform(query)

    search_time = time.perf_counter() - start

    evaluation = pt.Evaluate(
        results,
        qrels,
        metrics=[
            "map",
            "recip_rank",
            "P.5",
            "P.10",
            "ndcg_cut.10"
        ]
    )

    bm25_tuning_results.append({
        "k1": k1,
        "b": b,
        "k3": k3,
        "MAP": evaluation["map"],
        "MRR": evaluation["recip_rank"],
        "P@5": evaluation["P.5"],
        "P@10": evaluation["P.10"],
        "nDCG@10": evaluation["ndcg_cut.10"],
        "Search Time (s)": search_time
    })

    if count % 25 == 0:
        print(
            f"Fine search: {count}/{len(fine_configs)}"
        )


# ==========================================
# FINAL RESULTS
# ==========================================

bm25_final_df = pd.DataFrame(
    bm25_tuning_results
)

bm25_final_df = bm25_final_df.sort_values(
    by=["MAP", "nDCG@10", "MRR"],
    ascending=False
).reset_index(drop=True)


print("\n======================================")
print("FINAL TOP 20 BM25 CONFIGURATIONS")
print("======================================")

display(bm25_final_df.head(20))

Fine configurations: 1080
Fine search: 25/1080
Fine search: 50/1080
Fine search: 75/1080
Fine search: 100/1080
Fine search: 125/1080
Fine search: 150/1080
Fine search: 175/1080
Fine search: 200/1080
Fine search: 225/1080
Fine search: 250/1080
Fine search: 275/1080
Fine search: 300/1080
Fine search: 325/1080
Fine search: 350/1080
Fine search: 375/1080
Fine search: 400/1080
Fine search: 425/1080
Fine search: 450/1080
Fine search: 475/1080
Fine search: 500/1080
Fine search: 525/1080
Fine search: 550/1080
Fine search: 575/1080
Fine search: 600/1080
Fine search: 625/1080
Fine search: 650/1080
Fine search: 675/1080
Fine search: 700/1080
Fine search: 725/1080
Fine search: 750/1080
Fine search: 775/1080
Fine search: 800/1080
Fine search: 825/1080
Fine search: 850/1080
Fine search: 875/1080
Fine search: 900/1080
Fine search: 925/1080
Fine search: 950/1080
Fine search: 975/1080
Fine search: 1000/1080
Fine search: 1025/1080
Fine search: 1050/1080
Fine search: 1075/1080

FINAL TOP 20 BM25 CONFIGUR

,k1,b,k3,MAP,MRR,P@5,P@10,nDCG@10,Search Time (s)
0,3.00,0.76,8,0.323255,0.546081,0.334222,0.244889,0.349440,4.570166
1,3.00,0.76,10,0.323236,0.546082,0.334222,0.244889,0.349426,3.777497
2,3.00,0.76,9,0.323234,0.546082,0.334222,0.244889,0.349426,4.448965
3,3.00,0.76,11,0.323194,0.546082,0.334222,0.244889,0.349416,4.606742
4,3.00,0.76,12,0.323179,0.546083,0.334222,0.244889,0.349416,3.858309
5,2.99,0.75,2,0.322912,0.543424,0.334222,0.244444,0.349094,3.566436
6,2.95,0.76,10,0.322871,0.546295,0.333333,0.245333,0.349473,3.695580
7,2.90,0.80,1,0.322859,0.542060,0.333333,0.246222,0.350761,4.287584
8,2.95,0.76,11,0.322850,0.546296,0.333333,0.245333,0.349473,4.406324
9,2.95,0.76,12,0.322839,0.546296,0.333333,0.245333,0.349473,3.600879


In [ ]:
best_bm25 = bm25_final_df.iloc[0]

print("================================")
print("BEST BM25 CONFIGURATION")
print("================================")

print("k1      :", best_bm25["k1"])
print("b       :", best_bm25["b"])
print("k3      :", best_bm25["k3"])

print("\nRetrieval Performance")
print("---------------------")
print("MAP     :", best_bm25["MAP"])
print("MRR     :", best_bm25["MRR"])
print("P@5     :", best_bm25["P@5"])
print("P@10    :", best_bm25["P@10"])
print("nDCG@10 :", best_bm25["nDCG@10"])

print("\nSearch Time")
print("-----------")
print(best_bm25["Search Time (s)"])

BEST BM25 CONFIGURATION
k1      : 3.0
b       : 0.76
k3      : 8.0

Retrieval Performance
---------------------
MAP     : 0.32325497214311216
MRR     : 0.5460812363529111
P@5     : 0.33422222222222253
P@10    : 0.24488888888888916
nDCG@10 : 0.3494399696946975

Search Time
-----------
4.570165553999686


In [ ]:
bm25_eval = pt.Evaluate(
    bm25_results,
    qrels,
    metrics=[
        "map",
        "recip_rank",
        "P.5",
        "P.10",
        "ndcg_cut.10"
    ]
)

bm25_eval

{'map': 0.31570062120604253,
 'recip_rank': 0.5442072803431575,
 'P.5': 0.329777777777778,
 'P.10': 0.2373333333333335,
 'ndcg_cut.10': 0.3435079529364762}

In [ ]:
bm25_results_table = pd.DataFrame([{
    "Model": "BM25",
    "MAP": bm25_eval["map"],
    "MRR": bm25_eval["recip_rank"],
    "P@5": bm25_eval["P.5"],
    "P@10": bm25_eval["P.10"],
    "nDCG@10": bm25_eval["ndcg_cut.10"],
    "Index Time (s)": index_time,
    "Search Time (s)": search_time
}])

bm25_results_table

,Model,MAP,MRR,P@5,P@10,nDCG@10,Index Time (s),Search Time (s)
0,BM25,0.315701,0.544207,0.329778,0.237333,0.343508,3.843604,3.772898


In [ ]:
bm25_results_table

,Model,MAP,MRR,P@5,P@10,nDCG@10,Index Time (s),Search Time (s)
0,BM25,0.315701,0.544207,0.329778,0.237333,0.343508,3.843604,3.772898


What is the Issue?

In [ ]:
bm25_pairs = set(
    zip(bm25_results["qid"], bm25_results["docno"])
)

qrel_pairs = set(
    zip(qrels["qid"], qrels["docno"])
)

overlap = bm25_pairs & qrel_pairs

print("BM25 pairs:", len(bm25_pairs))
print("Qrels pairs:", len(qrel_pairs))
print("Matching pairs:", len(overlap))

BM25 pairs: 188975
Qrels pairs: 1837
Matching pairs: 1746


In [ ]:
print("BM25 queries:", bm25_results["qid"].nunique())
print("Qrel queries:", qrels["qid"].nunique())

BM25 queries: 225
Qrel queries: 225


In [ ]:
print("BM25 top 20 for query 001")

display(
    bm25_results[
        bm25_results["qid"] == 1
    ][
        ["qid", "docno", "rank", "score"]
    ].head(20)
)

BM25 top 20 for query 001


,qid,docno,rank,score


In [ ]:
print(bm25_results["qid"].unique()[:20])
print(qrels["qid"].unique()[:20])

['1' '2' '3' '4' '5' '6' '7' '8' '9' '10' '11' '12' '13' '14' '15' '16'
 '17' '18' '19' '20']
['1' '2' '3' '4' '5' '6' '7' '8' '9' '10' '11' '12' '13' '14' '15' '16'
 '17' '18' '19' '20']


In [ ]:
print(query["qid"].unique()[:30])

['1' '2' '3' '4' '5' '6' '7' '8' '9' '10' '11' '12' '13' '14' '15' '16'
 '17' '18' '19' '20' '21' '22' '23' '24' '25' '26' '27' '28' '29' '30']


In [ ]:
print("Number of queries:", query["qid"].nunique())
print("Number of qrel queries:", qrels["qid"].nunique())

print("\nQueries in qrels but NOT in query file:")
print(sorted(
    set(qrels["qid"]) - set(query["qid"]),
    key=int
))

Number of queries: 225
Number of qrel queries: 225

Queries in qrels but NOT in query file:
[]


In [ ]:
display(qrels[qrels["qid"] == "1"])

,qid,docno,label
0,1,184,2
1,1,29,2
2,1,31,2
3,1,12,3
4,1,51,3
5,1,102,3
6,1,13,4
7,1,14,4
8,1,15,4
9,1,57,2


In [ ]:
display(
    bm25_results[
        bm25_results["qid"] == "1"
    ][["qid", "docno", "rank", "score"]].head(20)
)

,qid,docno,rank,score
0,1,51,0,29.524889
1,1,486,1,28.757999
2,1,184,2,25.137115
3,1,12,3,24.962690
4,1,878,4,22.776485
5,1,665,5,19.851052
6,1,746,6,19.037803
7,1,573,7,19.034796
8,1,78,8,17.471362
9,1,141,9,17.433647


In [ ]:
print(qrels["label"].unique())

[ 2  3  4 -1  1]
